# Izrada aplikacija za generiranje slika (OpenAI)

LLM-ovi ne služe samo za generiranje teksta. Također možete generirati slike na temelju tekstualnih opisa. Slike kao modalitet korisne su u području medicinske tehnologije, arhitekture, turizma, razvoja igara, marketinga i drugih. U ovoj lekciji radimo s današnjim **GPT Image** modelima na OpenAI platformi.

## Ciljevi učenja

Na kraju ove lekcije moći ćete:

- Objasniti što je generiranje slika i gdje je korisno.
- Razumjeti obitelj modela `gpt-image` i kako se razlikuje od starih DALL·E modela.
- Izraditi aplikaciju za generiranje slika i uređivati slike.

## Što je generiranje slika?

Modeli za generiranje slika stvaraju slike na temelju tekstualnih uputa. Moderni modeli kao što je `gpt-image` tijekom treniranja uče odnos između teksta i slika, zatim iterativno pretvaraju slučajni šum u sliku koja odgovara vašoj uputi.

Dvije poznate obitelji modela za slike su:

- **`gpt-image` (OpenAI)** — trenutačna generacija korištena u ovoj lekciji. Podržava generiranje slike iz teksta i uređivanje slika (popunjavanje s maskom).
- **Midjourney** — popularan model treće strane sa svojom uslugom i radnim tijekovima baziranim na Discordu.

> Stariji OpenAI modeli za slike — **DALL·E 2** i **DALL·E 3** — su zastarjeli. DALL·E 3 više nije dostupan za nove implementacije, a značajke poput `create_variation` postojale su samo u DALL·E 2. Za nove aplikacije koristite `gpt-image` modele.

> **Važno:** `gpt-image` modeli vraćaju generiranu sliku kao **base64** (`b64_json`), a ne kao URL. Vaš kod dekodira base64 niz u bajtove i sprema sliku — nema URL-a slike za preuzimanje.


## Izgradnja vaše prve aplikacije za generiranje slika

Pa što je potrebno za izgradnju aplikacije za generiranje slika? Trebate sljedeće biblioteke:

- **python-dotenv**, preporučuje se da koristite ovu biblioteku za čuvanje vaših tajni u *.env* datoteci odvojeno od koda.
- **openai**, ovu biblioteku ćete koristiti za interakciju s OpenAI API-jem.
- **pillow**, za rad sa slikama u Pythonu.


1. Kreirajte datoteku *.env* sa sljedećim sadržajem:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Sakupite gore navedene biblioteke u datoteku nazvanu *requirements.txt* na sljedeći način:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Zatim, kreirajte virtualno okruženje i instalirajte biblioteke:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Za Windows koristite sljedeće naredbe za stvaranje i aktivaciju vašeg virtualnog okruženja:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Dodajte sljedeći kod u datoteku nazvanu *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Kreiraj OpenAI objekt (čita OPENAI_API_KEY iz tvoje .env datoteke)
    client = openai.OpenAI()


    try:
        # Kreiraj sliku koristeći API za generiranje slika
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Unesi svoj tekst upita ovdje
            size='1024x1024',
            n=1
        )
        # Postavi direktorij za spremljenu sliku
        image_dir = os.path.join(os.curdir, 'images')

        # Ako direktorij ne postoji, kreiraj ga
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicijaliziraj putanju slike (napomena: format datoteke treba biti png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image modeli vraćaju sliku kao base64 (b64_json), ne kao URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Prikaži sliku u zadanoj aplikaciji za pregled slika
        image = Image.open(image_path)
        image.show()

    # uhvati iznimke
    except openai.BadRequestError as err:
        print(err)

    ```

Hajde da objasnimo ovaj kod:

- Prvo, uvozimo biblioteke koje su nam potrebne, uključujući OpenAI biblioteku, dotenv biblioteku, base64 modul i Pillow biblioteku.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Nakon toga, kreiramo klijenta, koji učitava API ključ iz vašeg ``.env`` fajla.

    ```python
    # Stvori OpenAI objekt
    client = openai.OpenAI()
    ```

- Sljedeće, generiramo sliku:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Unesite vaš tekst upita ovdje
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modeli vraćaju sliku kao **base64** niz u `data[0].b64_json`. Dekodiramo ga u bajtove i zapisujemo u datoteku — ne postoji URL za preuzimanje.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Na kraju, otvaramo sliku i koristimo standardni preglednik slika za prikaz:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Više detalja o generiranju slike

Pogledajmo parametre `images.generate`:

- **model**, je model slike koji se koristi (na primjer `gpt-image-1`).
- **prompt**, je tekstualni upit korišten za generiranje slike. Ovdje je to "Zec na konju, drži lizalicu, na maglovitoj livadi gdje rastu narcisi".
- **size**, je veličina generirane slike (na primjer `1024x1024`, `1536x1024`, `1024x1536` ili `"auto"`).
- **n**, je broj generiranih slika. Ovdje generiramo jednu.

> Modeli slika ne prihvaćaju parametar `temperature` — to je kontrola za generiranje teksta. Za raznolikost, pozovite API ponovo; za manju raznolikost, učinite vaš upit preciznijim.

## Dodatne mogućnosti generiranja slika

Vidjeli ste kako generirati sliku sa samo nekoliko linija Pythona. `gpt-image` modeli također mogu **uređivati** postojeću sliku. Pružajući sliku, neobaveznu **masku** (koja označava područje za promjenu), i upit, možete promijeniti dio slike — na primjer, dodati šešir našem zecu.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# uređaji se također vraćaju kao base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Osnovna slika sadrži samo zeca; konačna slika ima šešir na zecu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
